In [1]:
import json
import sys
from rich import print as rprint
from pathlib import Path
import re

nb_dir = Path.cwd()

project_root = nb_dir.parent.parent

sys.path.insert(0, str(project_root))

In [2]:
db_people_file = Path(project_root / "data/from db/people-from-db.json")
people_file = Path(project_root / "data/people/prepped4loading/people.json")


with open(db_people_file, "r") as f:
   db_people = json.load(f)

with open(people_file, "r") as f:
   people = json.load(f)


db_lookup = {p["unified_id"]: p for p in db_people}
# rprint(dict(list(db_lookup.items())[:2]))

people_dict = {p["unified_id"]: p for p in people}
# rprint(dict(list(people_dict.items())[:2]))


In [ ]:
people_fixed_ids = {}
name_changes = {}


name_fields = ["family_name", "given_names", "name_prefix", "name_particles", "name_suffix", "single_name", "is_organisation"]

for unified_id, person in people_dict.items():
# for unified_id, person in list(people_dict.items())[:1]:
    person_id_old = person["person_id"]
    family_name = person["family_name"]
    given_names = person["given_names"]
    name_prefix = person["name_prefix"]
    name_particles = person["name_particles"]
    name_suffix = person["name_suffix"]
    single_name = person["single_name"]
    is_organisation = person["is_organisation"]


    data = db_lookup[unified_id]
    person_id_new = data["person_id"]

    if any(person[f] != data[f] for f in name_fields):
        name_changes[unified_id] = {
            "json": person,
            "db": data
        }
    else:
        people_fixed_ids[unified_id] = {
            **person,
            "person_id": person_id_new,
            "person_id_old": person_id_old,
        }

# with open("name_updates.json", "w") as f:
#     json.dump(name_changes, f, ensure_ascii=False, indent=2)

fixing_path = Path(project_root / "data/people/fixing_ids")
fixed_file = Path(project_root / fixing_path / "people_id_mapping.json")

# with open(fixed_file, "w") as f:
#     json.dump(people_fixed_ids, f, ensure_ascii=False, indent=2)


# rprint(f"""
# === LOG ===
# total people: {len(people)}
# total db people: {len(db_people)}

# updates: {len(name_changes)}
# ids_fixed: {len(people_fixed_ids)}

# lets do some math - sum of updates + fixed:
# {len(name_changes) + len(people_fixed_ids)}
# === DONE ===
# """)

=== LOG ===
total people: 7816
total db people: 7816

updates: 0
ids_fixed: 7816

lets do some math - sum of updates + fixed:
7816
=== DONE ===

In [ ]:
with open(fixed_file, "r") as f:
   people = json.load(f)
